# Analyse exploratoire — AlterVélo Réunion

Données : `data/stations_clean.csv` (sortie de `cleaning_data.py`).

Objectif : comprendre la structure du jeu (volumétrie, qualité, dynamiques temporelles et spatiales) avant l'entraînement XGBoost.

## 1. Imports et chargement

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.options.display.max_columns = 30

df = pd.read_csv('../data/stations_clean.csv', parse_dates=['timestamp'])
df = df.sort_values(['station_index', 'timestamp']).reset_index(drop=True)
print(df.shape)
df.head()

## 2. Vue d'ensemble

In [ ]:
print('Période :', df['timestamp'].min(), '->', df['timestamp'].max())
print('Nb stations :', df['station_index'].nunique())
print('Nb timestamps uniques :', df["timestamp"].nunique())
print('Lignes imputées :', f"{df['is_imputed'].mean()*100:.1f}%")
df.describe()

## 3. Qualité des données — imputations par station

In [ ]:
imp_by_station = df.groupby('station_name')['is_imputed'].mean().sort_values(ascending=False) * 100
fig, ax = plt.subplots(figsize=(8, 8))
imp_by_station.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('% lignes imputées')
ax.set_title('Part de mesures imputées par station')
plt.tight_layout()

## 4. Disponibilité globale — vélos vs docks

**Ce qu'on cherche** : la somme `vélos_dispo + docks_libres` devrait rester ≈ constante (la capacité physique du réseau ne change pas vite). Si l'une monte, l'autre descend symétriquement.

**Hypothèses à vérifier sur le graphe** :
- Voit-on une **anti-corrélation visible** entre les deux courbes (effet « bascule » : utilisateurs prennent vs déposent) ?
- Y a-t-il des **oscillations journalières** (creux le matin si les gens prennent les vélos pour aller travailler, remontée le soir) ?
- Y a-t-il des **discontinuités** (pannes, redéploiement de flotte par camion, jours de collecte interrompue) ?
- La somme totale est-elle **stable** ou dérive-t-elle (vélos cassés / retirés du réseau au fil du temps) ?

In [ ]:
global_ts = df.groupby('timestamp')[['num_vehicles_available', 'num_docks_available']].sum()
fig, ax = plt.subplots(figsize=(12, 4))
global_ts.plot(ax=ax)
ax.set_title('Total réseau : vélos disponibles vs docks libres')
ax.set_ylabel('Compte total')
plt.tight_layout()

## 5. Profil journalier moyen (heure locale)

**Ce qu'on cherche** : moyenner les vélos disponibles par heure révèle le **rythme typique** d'un jour. Comparer semaine vs week-end teste l'hypothèse d'usages distincts (pendulaire vs loisir).

**Hypothèses à vérifier** :
- En **semaine** : creux entre 7h-9h (départs domicile→travail, vélos quittent les stations résidentielles) puis remontée en soirée (18h-20h). Inverse possible pour les stations près des bureaux.
- Le **week-end** : pic plus tardif (10h-12h), plus étalé, sans creux net du matin.
- Le **différentiel semaine/week-end** est-il marqué ou les deux courbes se superposent-elles ? Faible différentiel = signal hebdomadaire faible = `lag_336` apportera peu.
- **Les nuits** (22h-5h) : ligne plate ? C'est exactement la zone que `fill_gaps` a comblée par forward-fill — si elle n'est pas plate, c'est suspect.

In [ ]:
df['hour'] = df['timestamp'].dt.hour
df['dow'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = df['dow'] >= 5

profile = df.groupby(['hour', 'is_weekend'])['num_vehicles_available'].mean().unstack()
profile.columns = ['Semaine', 'Week-end']
fig, ax = plt.subplots(figsize=(10, 4))
profile.plot(ax=ax, marker='o')
ax.set_title('Vélos disponibles moyens par station, selon l\'heure')
ax.set_xlabel('Heure locale (UTC+4)')
ax.set_ylabel('Vélos disponibles (moyenne)')
plt.tight_layout()

## 6. Heatmap heure × jour de la semaine

**Ce qu'on cherche** : la heatmap est la version 2D du profil précédent — elle révèle si le pattern journalier **change selon le jour** (mardi ≠ samedi) ou s'il est uniforme.

**Hypothèses à vérifier** :
- **Bandes verticales homogènes** (toutes les heures se ressemblent quel que soit le jour) → effet jour-de-la-semaine faible, lundi-vendredi indistinguables.
- **Contraste lundi-vendredi vs samedi-dimanche** → le modèle gagnera à utiliser `dow` comme feature et `lag_336` (1 semaine).
- **Cellules très sombres ou très claires isolées** → événements ponctuels (jour férié, panne) qui pollueraient la moyenne — à inspecter.
- Sur seulement **22 jours** (~3 semaines), chaque cellule jour×heure n'a que ~3 mesures moyennées : la heatmap sera **bruitée**. Méfiance avant de conclure à un pattern fort.

In [ ]:
heat = df.groupby(['dow', 'hour'])['num_vehicles_available'].mean().unstack()
heat.index = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(heat, cmap='viridis', ax=ax, cbar_kws={'label': 'vélos dispo (moy.)'})
ax.set_title('Disponibilité moyenne — heure × jour')
plt.tight_layout()

## 7. Stations les plus / moins fréquentées

**Ce qu'on cherche** : trier par `std` met en tête les stations dont le nombre de vélos varie le plus dans le temps — celles où une prévision a une vraie valeur ajoutée. Le tracé des séries des 3 premières montre **à quoi ressemble une station « difficile »**.

**Hypothèses à vérifier** :
- L'écart de `std` entre top et bottom est-il **grand** (rapport > 10×) ? Si oui, le réseau est très hétérogène et il faudra évaluer le modèle par segment, pas en moyenne globale.
- Les courbes des 3 stations top montrent-elles des **pics réguliers** (motif horaire / hebdo) ou un **bruit chaotique** ? Le premier cas est apprenable, le second non.
- Les stations top sont-elles plutôt des **hubs** identifiables par leur nom (gares, marchés, plages) ? C'est un sanity check : si une « station résidentielle perdue » se retrouve en tête, suspecter un capteur défectueux.

In [ ]:
rotation = df.groupby('station_name')['num_vehicles_available'].agg(['mean', 'std']).sort_values('std', ascending=False)
print('Top 10 stations à forte variabilité (rotation élevée) :')
rotation.head(10)

In [ ]:
top3 = rotation.head(3).index
fig, ax = plt.subplots(figsize=(12, 4))
for name in top3:
    sub = df[df['station_name'] == name].set_index('timestamp')['num_vehicles_available']
    sub.plot(ax=ax, label=name, alpha=0.8)
ax.legend(); ax.set_title('Séries temporelles — 3 stations à plus forte rotation')
plt.tight_layout()

## 8. Corrélations entre compteurs

**Ce qu'on cherche** : la matrice de corrélation entre les 5 compteurs détecte les **redondances** (deux variables qui mesurent la même chose) et les **fuites potentielles** vers la cible.

**Hypothèses à vérifier** :
- `num_vehicles_available` ↔ `num_docks_available` : on attend une **anti-corrélation forte** (proche de -1) puisqu'à capacité fixe, l'un est le complément de l'autre. Si la valeur est faible, c'est qu'il y a beaucoup de vélos / docks « disabled » qui flottent.
- `num_vehicles_available` ↔ `count_x2` : on attend une corrélation **très proche de +1** (les x2 sont la quasi-totalité des vélos disponibles). C'est exactement la fuite par proxy détectée pendant l'entraînement — elle doit être visible ici.
- `num_vehicles_disabled` et `num_docks_disabled` faiblement corrélés au reste → variables indépendantes, candidates à garder telles quelles.
- Toute corrélation **inattendue > 0.8** entre deux variables non liées par construction = piste à creuser.

In [ ]:
corr_cols = ['num_vehicles_available', 'num_vehicles_disabled',
             'num_docks_available', 'num_docks_disabled', 'count_x2']
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(df[corr_cols].corr(), annot=True, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Corrélations entre compteurs')
plt.tight_layout()

## 9. Autocorrélation (effet mémoire de la série)

**Ce qu'on cherche** : l'autocorrélation à lag *k* mesure à quel point la série « se ressemble » avec elle-même décalée de *k* pas. Pour notre granularité 30 min : k=48 → 24h ; k=336 → 1 semaine. Les pics d'autocorrélation valident **quels lags méritent d'être mis en feature** dans XGBoost.

**Hypothèses à vérifier** :
- **Décroissance rapide aux petits lags** (k=1 à 4) avec autocorrélation > 0.8 → le passé immédiat prédit bien le futur immédiat (la persistance est forte, baseline difficile à battre).
- **Pic local à k=48** (24h) → cycle journalier capté par la série, `lag_48` sera une feature utile.
- **Pic local à k=336** (1 semaine) → cycle hebdomadaire détectable. Si **absent ou plat**, alors `train_xgboost_weekly.py` n'apporte rien et on perd des données pour rien (rappel : `lag_336` coûte 7 jours d'historique).
- Une **autocorrélation qui reste élevée à tous les lags** = série très lente, faiblement informative. Une qui chute brutalement vers 0 = série quasi aléatoire, non apprenable.

⚠️ Cette cellule analyse **une seule station** (la première). Le pattern observé n'est pas forcément représentatif des autres — une boucle sur quelques stations contrastées (top et bottom de §7) donnerait une vue plus juste.

In [ ]:
from pandas.plotting import autocorrelation_plot

sample = df[df['station_index'] == df['station_index'].iloc[0]].set_index('timestamp')['num_vehicles_available']
fig, ax = plt.subplots(figsize=(10, 3))
autocorrelation_plot(sample, ax=ax)
ax.set_xlim(0, 400)  # 400 pas = ~8 jours
ax.set_title(f'Autocorrélation — station {sample.name}')
plt.tight_layout()

## Aller plus loin

Les données actuelles décrivent uniquement **l'état des stations** (vélos / docks à un instant). Pour améliorer la précision des prévisions à court, moyen et long terme, on pourrait enrichir le jeu avec :

- **Trajectoires des vélos** : positions GPS individuelles entre deux stations. Cela permettrait de prévoir les arrivées (« 12 vélos en route vers la station X dans 8 min »), donc d'anticiper le remplissage avant qu'il n'apparaisse dans `num_vehicles_available`. Très utile à **court terme** (< 30 min).
- **Localisation des stations** (lat/lon) couplée à un graphe de proximité : une station saturée déborde sur ses voisines. Modéliser ces transferts spatiaux apporte du signal à **moyen terme** (1-3h) — un GNN ou simplement des features d'agrégat de voisins.
- **Données météo** (température, pluie, vent, ensoleillement, alertes cycloniques) : à la Réunion les averses tropicales écroulent l'usage en quelques minutes, et la saison cyclonique crée des régimes complètement différents. Ces variables expliquent une grande part de la demande à **moyen et long terme** (jours, semaines, saisons).
- **Calendriers locaux** : jours fériés réunionnais, vacances scolaires, événements (Grand Raid, festivals), grèves de bus. Tout cela perturbe les cycles habituels que le modèle apprend des lags hebdomadaires.
- **Données socio-économiques** par quartier (densité, emplois, écoles, commerces) pour expliquer les profils de stations différents et améliorer la généralisation à de nouvelles stations.

Combiner ces sources transformerait le problème d'une simple série temporelle station par station en un **modèle spatio-temporel multi-source**, beaucoup plus robuste face aux situations inhabituelles.